# Introduction to HuggingFace for Molecular LLMs

---

## Learning Outcomes

By the end of this session, participants will be able to:
- Load datasets from HuggingFace using the `datasets` package
- Process the datasets that have been loaded
- Load and use tokenisers from HuggingFace

## credits

This notebooks is adapted from [Meg Hedge's ECCB2026 tutorial](https://github.com/meghegde/genomic-llms-in-practice/tree/main),
with idea and inspirations from [Raphael Mourad's genomic LLM finetuning on the Galaxy Training Network](https://training.galaxyproject.org/training-material/topics/statistics/tutorials/genomic-llm-finetuning/tutorial.html#solution-16) and [Hari Ramadasan's ELIXIR ML/AI 2026 workshop materials](https://github.com/hramdasan/elixir_summerschool_26/tree/main)

## Install and import required packages

> the installation should not be necessary on google colab

Uncomment and run the cell below if needed.

In [121]:
#!pip install transformers datasets pandas transformers

*N.B.: If you perform installs, restart the runtime between the installs and imports*

In [1]:
import pandas as pd
import torch
from datasets import load_dataset
import transformers
from transformers import AutoTokenizer, AutoConfig, AutoModel

# Loading a dataset

HuggingFace hosts a large number of datasets that can be used to train, explore, and evaluate our models.

We are going to play with [InstaDeepAI/nucleotide_transformer_downstream_tasks_revised](https://huggingface.co/datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks_revised) which contains train and test data for 18 classification tasks on DNA data.

You can read more on the data and tasks by clicking on the link above.


In [16]:
%%time
dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks_revised")
print(dataset)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 493242
    })
    test: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 38822
    })
})
CPU times: user 114 ms, sys: 13.4 ms, total: 128 ms
Wall time: 651 ms


This gives us a Huggingface DatasetDict, containing two Huggingface Dataset objects, one for the train and one for the test split.

In [17]:
train_dataset = dataset["train"]
print(train_dataset)

Dataset({
    features: ['sequence', 'name', 'label', 'task'],
    num_rows: 493242
})


A Huggingface Dataset can be converted into a Pandas DataFrame.

In [18]:
train_df = train_dataset.to_pandas()
train_df.head()

,sequence,name,label,task
0,ATATTTTTCGGTGTTTTTTTAAAATCCAGAAAAGGTTAATTGTTTT...,chr8:16774000-16775000|0,0,H2AFZ
1,AAGTATAGTCATAAAGGTGGGGGATTTAATCATGTTCCAGAATGAT...,chr12:12784548-12785548|1,1,H2AFZ
2,CCCCAGCATGGACTTGCAGGGGCGCCTGGTGGGAGAGGGGTGTGGA...,chr1:1376174-1377174|1,1,H2AFZ
3,GCCTGTAATCCCAGCACTTTGAGAGGCCGGGGCGGGCAGATCACGA...,chr17:78711000-78712000|0,0,H2AFZ
4,CTGCGGCTTTTGGAGGGAGCTTGGCTTGGAACCCAGGGTTCTCTGA...,chr8:127981888-127982888|1,1,H2AFZ


You can then do all the usual Pandas manipulations, which can be helpful for inspecting the dataset.

In [20]:
train_df[train_df.task=="H2AFZ"]

,sequence,name,label,task
0,ATATTTTTCGGTGTTTTTTTAAAATCCAGAAAAGGTTAATTGTTTT...,chr8:16774000-16775000|0,0,H2AFZ
1,AAGTATAGTCATAAAGGTGGGGGATTTAATCATGTTCCAGAATGAT...,chr12:12784548-12785548|1,1,H2AFZ
2,CCCCAGCATGGACTTGCAGGGGCGCCTGGTGGGAGAGGGGTGTGGA...,chr1:1376174-1377174|1,1,H2AFZ
3,GCCTGTAATCCCAGCACTTTGAGAGGCCGGGGCGGGCAGATCACGA...,chr17:78711000-78712000|0,0,H2AFZ
4,CTGCGGCTTTTGGAGGGAGCTTGGCTTGGAACCCAGGGTTCTCTGA...,chr8:127981888-127982888|1,1,H2AFZ
...,...,...,...,...
29995,TATTAATATCTGACTAAGGAATGGGCTCTTTCTTCTCCCACTCCTA...,chr3:139204486-139205486|1,1,H2AFZ
29996,ATTCAGGTTTCATTTGTTAAATCTGCAAGCTCTGTATGCTGTCTTT...,chr2:145237926-145238926|1,1,H2AFZ
29997,GATATGAACTTTGTTGACGAATGTGTTCTCTGCACAGCTAACCCAG...,chr14:101051000-101052000|0,0,H2AFZ
29998,GTTATTAGGCACAAGACACTTCTTTTAGGCACAAGACACTTCTTTT...,chr16:71240000-71241000|0,0,H2AFZ


In [122]:
train_df[train_df.task=="H2AFZ"].label.value_counts()

,count
label,
0,15003
1,14997


In [123]:
train_df.task.value_counts()

,count
task,
H2AFZ,30000
H3K27ac,30000
H3K27me3,30000
H3K36me3,30000
H3K4me1,30000
H3K4me2,30000
H4K20me1,30000
promoter_all,30000
enhancers_types,30000


The Huggingface Dataset has its own functions that let you manipulate it directly.


*  `train_test_split` : can be used to further divide the dataset (into a training and a validation set for example):

In [23]:
# produces another DatasetDict, containing two Datasets - one "train" and the other "test"
split_dataset = train_dataset.train_test_split(test_size=0.1)
split_dataset

DatasetDict({
    train: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 443917
    })
    test: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 49325
    })
})



* `select`: to extract specific rows by index

In [24]:
small_dataset = train_dataset.select(range(50))
print(small_dataset)
print(len(small_dataset))

Dataset({
    features: ['sequence', 'name', 'label', 'task'],
    num_rows: 50
})
50


* `filter`: to extract rows based on some property





In [126]:
filtered_train_dataset = train_dataset.filter(lambda x: x["task"]=="H2AFZ")
print(filtered_train_dataset)
filtered_train_dataset.to_pandas().task.value_counts()

Filter:   0%|          | 0/493242 [00:00<?, ? examples/s]

Dataset({
    features: ['sequence', 'name', 'label', 'task'],
    num_rows: 30000
})


,count
task,
H2AFZ,30000


---

Actually, we are going to generate right now a version of the dataset focused on the H2AFZ task (which is about  predicting whether it contains a binding site for the H2AFZ histone variant)

In [128]:
dataset = dataset.filter(lambda x: x["task"]=="H2AFZ")
dataset

Filter:   0%|          | 0/493242 [00:00<?, ? examples/s]

Filter:   0%|          | 0/38822 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 3000
    })
})

# Tokeniser

* The tokeniser is used to prepare the data for model input.
* It breaks a sequence into smaller chunks or "tokens".
* A **tokenizer is specific to a model**

Related Huggingface pages:
* https://huggingface.co/docs/transformers/en/main_classes/tokenizer
* https://huggingface.co/learn/llm-course/chapter2/4

## Loading a tokenizer

* This tokeniser splits sequences into 6-mers - chunks of 6 nucleotides - when possible, otherwise tokenising each nucleotide separately.
* This tokenizer has a vocabulary size of 4105. The inputs of the model are then of the form:
```
<CLS> <ACGTGT> <ACGTGC> <ACGGAC> <GACTAG> <TCAGCA>
```
* The tokenised sequences have a maximum length of 1,000.

Related Huggingface page: https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species


In [130]:
%%time
MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

CPU times: user 35.1 ms, sys: 1.08 ms, total: 36.2 ms
Wall time: 206 ms


In general the code above is fine, but just to make things more explicit we are going to repeat the above with some parameters:

In [133]:
tokeniser = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    model_max_length=1000,
    padding_side="right",
    use_fast=True,
    trust_remote_code=True,
)



* `model_max_length=1000`: Sets the maximum length of the tokenized sequences. Sequences longer than this will be truncated, and shorter ones will be padded.
* `padding_side="right"`: Specifies that padding should be added to the right side of the sequences. This ensures that all sequences in a batch have the same length.
* `use_fast=True`: Enables the use of the fast tokenizer implementation, which is optimized for speed and is suitable for most use cases.
* `trust_remote_code=True`: Allows the tokenizer to execute custom code from the model repository, which may be necessary for some models that require specific preprocessing steps.

*from [galaxy's genomic llm finetuning](https://training.galaxyproject.org/training-material/topics/statistics/tutorials/genomic-llm-finetuning/tutorial.html#solution-16)*

## Tokenize an example sequence

Let's get an example sequence

In [106]:
sequence = "ATATTTTTCGGTGTTTTTTTAAAATCCAGATT"
print(sequence)
print(len(sequence))

ATATTTTTCGGTGTTTTTTTAAAATCCAGATT
32


By passing the sequence into the tokeniser, we get the input ids and attention mask.

In [107]:
tok = tokeniser(sequence)
print(tok)

{'input_ids': [3, 283, 1475, 3419, 1286, 1682, 4103, 4103], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [108]:
token_ids = tok["input_ids"]
print(token_ids)

[3, 283, 1475, 3419, 1286, 1682, 4103, 4103]


In [109]:
attention_mask = tok["attention_mask"]
print(attention_mask)

[1, 1, 1, 1, 1, 1, 1, 1]


By using the `tokenize` function, we get the tokens themselves.

In [110]:
tokens = tokeniser.tokenize(sequence)
print(tokens)

['ATATTT', 'TTCGGT', 'GTTTTT', 'TTAAAA', 'TCCAGA', 'T', 'T']


To decode the sequence, we can put the input ids into the `decode` function.

In [111]:
decoded = tokeniser.decode(token_ids)
print(decoded)

<cls> ATATTT TTCGGT GTTTTT TTAAAA TCCAGA T T


In practice, models often require their input to be of a fixed length.
To do achieve this we **pad** the sequences:

In [113]:
tok = tokeniser(sequence, padding = 'max_length')
print(tok['input_ids'])
print(tok['attention_mask'])

[3, 283, 1475, 3419, 1286, 1682, 4103, 4103, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [116]:
decoded = tokeniser.decode(tok['input_ids'])
print(decoded)

<cls> ATATTT TTCGGT GTTTTT TTAAAA TCCAGA T T <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <

You can learn more about the tokenizer object from itself:

In [120]:
print( "tokenizer vocabulary size", tokeniser.vocab_size )
print( "tokenizer special tokens", tokeniser.special_tokens_map )

tokenizer vocabulary size 4107
tokenizer special tokens {'unk_token': '<unk>', 'pad_token': '<pad>', 'cls_token': '<cls>', 'mask_token': '<mask>'}


### Exercise: Tokenise the example sequence for DNABERT-2
Huggingface page for [zhihan1996/DNABERT-2-117M](https://huggingface.co/zhihan1996/DNABERT-2-117M)

In [ ]:
# Get the model name
MODEL_NAME =

In [ ]:
# Load the tokeniser
# Use a maximum length of 512bp
# Pad on the left side only
tokeniser =

In [ ]:
# Get the tokenised output

In [ ]:
# Get the input_ids

In [ ]:
# Decode the input_ids

#### Answer

In [134]:
MODEL_NAME = "zhihan1996/DNABERT-2-117M"

In [135]:
%%time
# Load the tokeniser
# Use a maximum length of 512bp
# Pad on the left side only
tokeniser = transformers.AutoTokenizer.from_pretrained(
    MODEL_NAME,
    model_max_length=512,
    padding_side="left",
    use_fast=True,
    trust_remote_code=True,
)

CPU times: user 43.8 ms, sys: 1.99 ms, total: 45.8 ms
Wall time: 226 ms


In [136]:
# Get the tokenised output
tokens = tokeniser.tokenize(sequence)
print(tokens)

['A', 'TATTTT', 'TCGGTG', 'TTTTTT', 'TAAAA', 'TC', 'CAGATT']


In [137]:
# Get the input_ids
input_ids = tokeniser(sequence)["input_ids"]
print(input_ids)

[1, 5, 155, 1848, 122, 58, 16, 410, 2]


In [138]:
# Decode the input_ids
decoded = tokeniser.decode(input_ids)
print(decoded)

[CLS] A TATTTT TCGGTG TTTTTT TAAAA TC CAGATT [SEP]


## Tokenise several sequences

We will come back to [InstaDeepAI/nucleotide-transformer-v2-50m-multi-species](https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species)

In [42]:
MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"

tokeniser = transformers.AutoTokenizer.from_pretrained(
    MODEL_NAME,
    model_max_length=512,
    padding_side="right",
    use_fast=True,
    trust_remote_code=True,
)

### Tokenize a list of sequences

Let's select the first five sequences in the dataset as our batch.

In [139]:
batch = train_dataset["sequence"][:5]
print(len(batch))
print(batch)

5
['ATATTTTTCGGTGTTTTTTTAAAATCCAGAAAAGGTTAATTGTTTTTTATATTGACCATGTTTTCTGTAAATTCTTTCATGCTTTATCTTTTTTCTTTGCATTATCCGAGTTATTTCCAGCCTGTTTGTCATTGGTTTAATTTTCAACAATGCCTACCTCACTATTTTTTATTAGTTCCAATATTATGTCTTTTTATACTCAATTCATTTCCCTAGTTCTCCCATACTATTCTCTGGTCTGTTATCTTTTCATCTTTGAGTTTTGGTTTTATATAGTCTATACACTTTATAATTTCTTTAAAAATGCAAAGCATTAGTTGTGAATGTTTTTCTTTTGTTTTCTAGGGCGTGTTCTCATTTAGAGTACTACGCATGAGAAGGTCATTTATTTCATTTTTTAAATAATCATAGGAACTTCTCATAGTTGTTATCCTGTCTCTTCTCATTTCACTCATATTAGACAAGGCAGCTCTTCTCTTTGCCCATGTGTGAGCTCTCCTTGAATCTCTTCTTATCTGCGATCTAAGAGTGTTGTCTTACCTTCAGATGGTCAACGTGAAGAATGGGTATTGATAGGTCACACTCACTCTGCTCTTTGTTTATCTGAGATTGAGAAGCTAGAGAGAGGAAGCTATGGGCAGTTAGGAAGAAGTTGGTCTCTTCTCTATTTTCCTATTGATTTTATAAAATCCTGTTCTGTATTTTGTTTTTTCTAGTGAAATGTTGACTTCTCTCCTTAATTCAGAGTGAGCTGCTGAATCTTCCCTCTACAAAATATCACTCATTCTTACTAGGCTTTATTCCTCTTTCCACTGAATATTGAGCTTCTAAGTTCAAATGGGAGTCTTTCAGGCTAGCCATTAACTTTTCCTGTTCTGTAAATGCCAAGATAAAATATTATGAGAGATTAAATTAGTAAATCTAGTTAGATTTTTTCTTATAATGAAAATACAAGCTGGAAAATTGGAGAATGCTGTACAACCCTTCCAAACTCCA

The tokeniser can natively handle batches.

In [140]:
encoded_batch = tokeniser(batch)
print(encoded_batch)

{'input_ids': [[1, 5, 155, 1848, 122, 58, 82, 85, 15, 453, 87, 1337, 17, 249, 1354, 244, 29, 224, 29, 276, 122, 2400, 3648, 17, 381, 3331, 341, 127, 101, 399, 27, 238, 114, 190, 854, 273, 2532, 1892, 167, 46, 1498, 51, 942, 2468, 104, 186, 178, 220, 24, 167, 734, 3085, 2070, 46, 2666, 219, 1871, 47, 1379, 49, 754, 31, 135, 1261, 167, 87, 37, 354, 163, 63, 94, 282, 37, 159, 23, 259, 127, 887, 1345, 20, 434, 57, 1533, 237, 77, 177, 271, 432, 1890, 147, 225, 112, 471, 1020, 95, 3498, 19, 816, 25, 271, 81, 54, 93, 172, 2965, 697, 64, 521, 168, 1063, 2899, 43, 1462, 411, 63, 178, 149, 687, 126, 73, 145, 105, 50, 534, 772, 112, 77, 1147, 316, 271, 40, 155, 929, 1302, 58, 1552, 28, 155, 509, 37, 327, 48, 226, 596, 453, 2726, 62, 119, 47, 372, 1572, 211, 432, 81, 37, 269, 43, 559, 82, 119, 43, 831, 79, 31, 373, 33, 161, 289, 37, 19, 313, 91, 705, 28, 318, 76, 1243, 147, 1505, 364, 24, 120, 88, 77, 2165, 81, 3542, 196, 62, 618, 33, 223, 28, 196, 232, 131, 415, 12, 8, 2], [1, 9, 493, 67, 38, 885

The `tokenize` function does not have a native batch mode; instead, it process one string at a time.

In [ ]:
tokens = tokeniser.tokenize(batch)
print(tokens)

TypeError: unhashable type: 'list'

Instead:

In [141]:
tokens = [tokeniser.tokenize(seq) for seq in batch]
print(tokens)

[['A', 'TATTTT', 'TCGGTG', 'TTTTTT', 'TAAAA', 'TCCA', 'GAAAA', 'GG', 'TTAATT', 'GTTTT', 'TTATATT', 'GA', 'CCATG', 'TTTTCTG', 'TAAATT', 'CTT', 'TCATG', 'CTT', 'TATC', 'TTTTTT', 'CTTTGCA', 'TTATCC', 'GA', 'GTTATT', 'TCCAGCC', 'TGTTTG', 'TCATT', 'GGTT', 'TAATTTT', 'CAA', 'CAATG', 'CCTA', 'CCTCA', 'CTATTTT', 'TTATTA', 'GTTCCAA', 'TATTATG', 'TCTTTT', 'TATA', 'CTCAATT', 'CATT', 'TCCCTA', 'GTTCTCC', 'CATA', 'CTATT', 'CTCTG', 'GTCTG', 'TTA', 'TCTTTT', 'CATCTT', 'TGAGTTTT', 'GGTTTTA', 'TATA', 'GTCTATA', 'CACTT', 'TATAATT', 'TCTT', 'TAAAAATG', 'CAAA', 'GCATTA', 'GTT', 'GTGAA', 'TGTTTT', 'TCTTTT', 'GTTTT', 'CTA', 'GGGC', 'GTGTT', 'CTCA', 'TTTA', 'GAGTA', 'CTA', 'CGCA', 'TGA', 'GAAGG', 'TCATT', 'TATTTCA', 'TTTTTTAAA', 'TAA', 'TCATA', 'GGAA', 'CTTCTCA', 'TAGTT', 'GTTA', 'TCCTG', 'TCTCTT', 'CTCATT', 'TCACTCA', 'TATTA', 'GACAA', 'GGCA', 'GCTCTT', 'CTCTTTG', 'CCCA', 'TGTGTGA', 'GC', 'TCTCCTT', 'GAA', 'TCTCTT', 'CTTA', 'TCTG', 'CGA', 'TCTAA', 'GAGTGTT', 'GTCTTA', 'CCTT', 'CAGATG', 'GTCAA', 'CGTGAA', 'G

### Tokenise a whole dataset

We could extract all the sequences in our Dataset a to a list and tokenize them as we have just done, but that is not the most efficient way to do things! When working with large datasets or tokenising multiple columns, it is preferable to use the `.map` function.

This is a built-in function of the Huggingface dataset object, and efficiently handles batches and multiprocessing. This approach avoids loading all sequences into memory at once.

However, this is not the most efficient way to do things! When working with large datasets or tokenising multiple columns, it is preferable to use the `.map` function.

This is a built-in function of the Huggingface dataset object, and efficiently handles batches and multiprocessing. This approach avoids loading all sequences into memory at once.

In [144]:
train_dataset = dataset['train']

We first define a tokenise function.

In [74]:
def tokenise_batch(batch):
    return tokeniser(
        batch["sequence"], # Select only the sequence column
        padding=True,
        truncation=True,
        max_length=512,
    )

We then apply this to the dataset using `.map`.

In [145]:
%%time
# extract the train dataset
train_dataset = dataset['train']

# tokenize the train dataset is batches of 64
tokenised_dataset = train_dataset.map(tokenise_batch,
                                      batched=True, batch_size=64)
print(tokenised_dataset)

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Dataset({
    features: ['sequence', 'name', 'label', 'task', 'input_ids', 'attention_mask'],
    num_rows: 30000
})
CPU times: user 17.4 s, sys: 232 ms, total: 17.6 s
Wall time: 14.1 s


### Exercise: Tokenise all the sequences in the test split of the dataset

In [ ]:
# Get only the test section of the dataset
test_dataset = # Your code here
print(test_dataset)

In [ ]:
# Tokenise all the sequences in the column

In [ ]:
# Decode the sequences

#### Answer

In [146]:
# Get only the test section of the dataset
test_dataset = dataset["test"]
print(test_dataset)

Dataset({
    features: ['sequence', 'name', 'label', 'task'],
    num_rows: 3000
})


In [147]:
%%time
# Tokenise all the sequences in the column
tokenised_dataset = test_dataset.map(tokenise_batch, batched=True, batch_size=64)
print(tokenised_dataset)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['sequence', 'name', 'label', 'task', 'input_ids', 'attention_mask'],
    num_rows: 3000
})
CPU times: user 1.77 s, sys: 21 ms, total: 1.79 s
Wall time: 1.57 s


---

Finally, it is also possible to tokenize both sets at once:

In [151]:
tokenised_dataset = dataset.map(tokenise_batch, batched=True, batch_size=64)


Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

### Decode several sequences

In [152]:
# Decode the sequences
tokens = list(tokenised_dataset['train']["input_ids"])[:10]
# Method 1: list`
decoded = tokeniser.batch_decode(tokens)
print(decoded)

['[PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [CLS] A TATTTT TCGGTG TTTTTT TAAAA TCCA GAAAA GG TTAATT GTTTT TTATATT GA CCATG TTTTCTG TAAATT CTT TCATG CTT TATC TTTTTT CTTTGCA TTATCC GA GTTATT TCCAGCC TGTTTG TCATT GGTT TAATTTT CAA CAATG CCTA CCTCA CTATTTT TTATTA GTTCCAA TATTATG TCTTTT TATA CTCAATT CATT TCCCTA GTTCTCC CATA CTATT CTCTG GTCTG TTA TCTTTT CATCTT TGAGTTTT GGTTTTA TATA GTCTATA CACTT TATAATT TCTT TAAAAATG CAAA GCATTA GTT GTGAA TGTTTT TCTTTT GTTTT CTA GGGC GTGTT CTCA TTTA GAGTA CTA CGCA TGA GAAGG TCATT TATTTCA TTTTTTAAA TAA TCATA GGAA CTTCTCA TAGTT GTTA TCCTG TCTCTT CTCATT TCACTCA TATTA GACAA GGCA GCTCTT CTCTTTG CCCA TGTGTGA GC TCTCCTT GAA TCTCTT CTTA TCTG CGA TCTAA GAGTGTT GTCTTA CCTT CAGATG GTCAA CGTGAA GAATGGG TATT GATAGG TCACA CTCA CTCTG CTCTT TGTTTA TCTGA GATT GAGAA GCTA GAGA GAGGAA GCTATG GGCA GTTA GGAAGAA GTTGG TCTCTT CTC TATTTT CCTATT GATTTTA TAAAA TCCTGTT CTG TATTTT GTTTTTT CTA GTGAAA TGTT GACTT CTCTCC

In [154]:
%%time
# Method 2: map
def decode_batch(batch):
  return {"tokens": tokeniser.decode(batch["input_ids"])}

decoded = tokenised_dataset.map(decode_batch, batched=True, batch_size=64)
print(decoded)
print(decoded['test']["tokens"])

DatasetDict({
    train: Dataset({
        features: ['sequence', 'name', 'label', 'task', 'input_ids', 'attention_mask', 'tokens'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['sequence', 'name', 'label', 'task', 'input_ids', 'attention_mask', 'tokens'],
        num_rows: 3000
    })
})
Column(['[PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [CLS] CTTCTA CTTAA CCTGTG GTTTAAA GTC GGCTG TGACC TCCTG TCC CCAGA GCTG TACGG CCCTG CAGTG GGTG GGAGGAA CTTGCA TAAA TATATATT TATG GACACA CAGTT TGCA TCAGAA CGTGTT TATA GAATGTG TTTTA TACC CGA TCGTG TGTG GTGTG CGTGA GGACAAA CTCC GCAGGGG CTGTGAA TCC CACTG GGAGG GCGG TGGG CCTG CAGCC TGAGGAA GGCTT GTGTG TCCTCA GTTAAAA CTGTG CATA TCGAAA TATATTTT GTTATT TAA GCCTG CGTCC CCGTA CC GCGTG CTTTATT TCTA TGA GTTTG CAGA GTTTG GTG GTCTC GCCA TCCTT GGCCCA CGTCC TGTA GTCA GCC CTGTG GAATCC GTGGA TATGAAAA GTTGG CCTT CCTTA TCCACA GGCTCC CC CGTAA GTC GGCCTT CCTTA TCCACA GGCTCC CCTG CTGAGGA TTCA TCCCA CCTT GGATG GAAAA TGCA GTTTT TGTG GGAGATG GAACTG GAGG GTA C